# Lecture 1 Exercises

## Linear Regression, Model Complexity, and Regularisation

This notebook accompanies the introductory machine learning lecture. It focuses on three connected ideas:

- deriving linear regression from a probabilistic model
- understanding underfitting and overfitting through polynomial models
- controlling model complexity with ridge regularisation

Suggested workflow:

1. Answer the written prompts in markdown cells.
2. Complete the `TODO` blocks in the code cells.
3. Re-run the analysis cells to inspect the behaviour of your models.


In [ ]:
import os
from pathlib import Path

cache_root = Path.cwd() / ".notebook_cache"
mpl_config_dir = cache_root / "matplotlib"
xdg_cache_dir = cache_root / "xdg"
mpl_config_dir.mkdir(parents=True, exist_ok=True)
xdg_cache_dir.mkdir(parents=True, exist_ok=True)

os.environ["MPLCONFIGDIR"] = str(mpl_config_dir)
os.environ["XDG_CACHE_HOME"] = str(xdg_cache_dir)

import numpy as np
import matplotlib.pyplot as plt

style_name = "seaborn-v0_8-whitegrid"
plt.style.use(style_name if style_name in plt.style.available else "default")
np.set_printoptions(precision=3, suppress=True)


def mse(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return np.mean((y_true - y_pred) ** 2)


def add_bias_column(x):
    x = np.asarray(x).reshape(-1, 1)
    return np.hstack([np.ones((x.shape[0], 1)), x])


def polynomial_design_matrix(x, degree):
    x = np.asarray(x).reshape(-1)
    return np.column_stack([x ** k for k in range(degree + 1)])


def make_linear_data(n=40, noise_std=0.35, seed=7):
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(-2.0, 2.0, size=n))
    y_true = 1.5 - 2.0 * x
    y = y_true + rng.normal(0.0, noise_std, size=n)
    return x, y, y_true


def make_curved_data(n=45, noise_std=0.18, seed=11):
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(-1.0, 1.0, size=n))
    y_true = np.sin(np.pi * x) + 0.3 * x ** 2
    y = y_true + rng.normal(0.0, noise_std, size=n)
    return x, y, y_true


In [ ]:
x_lin, y_lin, y_lin_true = make_linear_data()
x_curve, y_curve, y_curve_true = make_curved_data()

rng = np.random.default_rng(5)
indices = rng.permutation(len(x_curve))
train_idx = indices[:20]
valid_idx = indices[20:32]
test_idx = indices[32:]

x_train, y_train = x_curve[train_idx], y_curve[train_idx]
x_valid, y_valid = x_curve[valid_idx], y_curve[valid_idx]
x_test, y_test = x_curve[test_idx], y_curve[test_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(x_lin, y_lin, color="tab:blue", label="observed data")
axes[0].plot(x_lin, y_lin_true, color="black", linewidth=2, label="true line")
axes[0].set_title("Dataset A: approximately linear")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()

axes[1].scatter(x_train, y_train, color="tab:blue", label="train")
axes[1].scatter(x_valid, y_valid, color="tab:orange", label="validation")
axes[1].scatter(x_test, y_test, color="tab:green", label="test")
axes[1].plot(x_curve, y_curve_true, color="black", linewidth=2, label="true signal")
axes[1].set_title("Dataset B: curved relationship")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend()

plt.tight_layout()
plt.show()


## Exercise 1: Linear Regression as Maximum Likelihood

Assume the data follow the model

$$y_i = \theta_0 + \theta_1 x_i + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2).$$

When you switch to matrix notation, let $y = (y_1, \ldots, y_n)^T$ be the column vector of targets, and let $X$ be the design matrix whose $i$-th row is $(1, x_i)$.

Work through the following derivation.

1. Write down the likelihood $p(y_i \mid x_i, \theta_0, \theta_1, \sigma^2)$ for a single observation.
2. Write the likelihood for the full dataset assuming independent observations.
3. Take the logarithm and simplify the log-likelihood up to additive constants.
4. Show that maximising the log-likelihood is equivalent to minimising the residual sum of squares

$$\sum_{i=1}^n (y_i - \theta_0 - \theta_1 x_i)^2.$$

5. Rewrite the model in matrix form $y = X\theta + \varepsilon$ and derive the normal equation

$$X^T X \hat{\theta} = X^T y.$$

6. Under what condition is the closed-form estimator unique?


### Your derivation

Write your answer here. A good solution should connect the Gaussian noise assumption directly to squared-error minimisation.


## Exercise 2: Ordinary Least Squares in Code

Use the closed-form solution you derived above to fit the linear dataset.

Tasks:

- complete `fit_ols` using the normal equation
- complete `predict`
- run the analysis cell and compare your fitted parameters with the true line
- check whether the estimate is close to the generating parameters $(1.5, -2.0)$
- inspect the design matrix example below and explain how the polynomial basis changes the feature space


In [ ]:
example_x = np.array([-1.0, 0.0, 2.0])
print("Linear design matrix:")
print(add_bias_column(example_x))
print()
print("Polynomial design matrix of degree 3:")
print(polynomial_design_matrix(example_x, degree=3))


In [ ]:
def fit_ols(X, y):
    """Return the ordinary least squares estimate."""
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    # TODO: compute X.T @ X and X.T @ y
    # TODO: solve the linear system for theta_hat
    # Hint: prefer np.linalg.solve over an explicit matrix inverse.
    raise NotImplementedError("Replace this with your OLS solver.")


def predict(X, theta):
    X = np.asarray(X)
    theta = np.asarray(theta).reshape(-1)

    # TODO: return the predictions X @ theta
    raise NotImplementedError("Replace this with your prediction function.")


In [ ]:
X_lin = add_bias_column(x_lin)
theta_lin = fit_ols(X_lin, y_lin)
y_lin_hat = predict(X_lin, theta_lin)

print("Estimated parameters:", theta_lin)
print(f"Training MSE: {mse(y_lin, y_lin_hat):.4f}")
print("Expected: the fitted parameters should be close to [1.5, -2.0].")

plt.figure(figsize=(6, 4))
plt.scatter(x_lin, y_lin, color="tab:blue", label="observed data")
plt.plot(x_lin, y_lin_true, color="black", linewidth=2, label="true line")
plt.plot(x_lin, y_lin_hat, color="tab:red", linewidth=2, label="OLS fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Ordinary least squares on the linear dataset")
plt.legend()
plt.show()


## Exercise 3: Underfitting and Overfitting with Polynomial Features

We now move to the curved dataset. Fit polynomial regression models with degrees 1, 3, 9, and 15.

Tasks:

1. Fit each model using only the training split.
2. Compute both training MSE and validation MSE.
3. Plot the fitted curve for each degree on a dense grid of input points.
4. Decide which model underfits and which model overfits.
5. Explain why the model with the lowest training error may still be a poor choice.


In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x_train, y_train, color="tab:blue", label="train")
plt.scatter(x_valid, y_valid, color="tab:orange", label="validation")
plt.scatter(x_test, y_test, color="tab:green", label="test")
plt.plot(x_curve, y_curve_true, color="black", linewidth=2, label="true signal")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Polynomial regression dataset")
plt.legend()
plt.show()


In [ ]:
def fit_polynomial_model(x, y, degree):
    """Fit a polynomial model of the requested degree using OLS."""
    # TODO: build the design matrix and call fit_ols
    raise NotImplementedError("Fit a polynomial model of the chosen degree.")


degrees = [1, 3, 9, 15]
results = []

for degree in degrees:
    # TODO: fit the model on the training split
    # TODO: compute train and validation predictions
    # TODO: append a dictionary with degree, theta, train_mse, and valid_mse
    pass

assert results, "Fill in the loop above before continuing."
results


In [ ]:
grid = np.linspace(-1.0, 1.0, 400)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

for ax, result in zip(axes.flat, results):
    degree = result['degree']
    theta = result['theta']
    y_grid = polynomial_design_matrix(grid, degree) @ theta

    ax.scatter(x_train, y_train, color="tab:blue", label="train")
    ax.scatter(x_valid, y_valid, color="tab:orange", label="validation")
    ax.plot(grid, np.sin(np.pi * grid) + 0.3 * grid ** 2, color="black", linewidth=2, label="true signal")
    ax.plot(grid, y_grid, color="tab:red", linewidth=2, label="model")
    ax.set_title(
        f"degree = {degree} | train MSE = {result['train_mse']:.3f} | valid MSE = {result['valid_mse']:.3f}"
    )

for ax in axes[-1, :]:
    ax.set_xlabel("x")

for ax in axes[:, 0]:
    ax.set_ylabel("y")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02))
plt.tight_layout()
plt.show()

for result in results:
    print(
        f"degree = {result['degree']:>2} | train MSE = {result['train_mse']:.4f} | validation MSE = {result['valid_mse']:.4f}"
    )


### Reflection questions

- Which degree gives the smallest training error?
- Which degree gives the best validation error?
- Which model is clearly underfitting? Which one is overfitting?
- How do the fitted curves change as the model becomes more flexible?
- What would happen if you increased the number of training points?


## Exercise 4: Ridge Regularisation

High-degree polynomial models can have very large coefficients and unstable predictions. Ridge regression controls this by adding a penalty to the objective:

$$J(\theta) = ||y - X\theta||_2^2 + \lambda ||\theta||_2^2.$$

Tasks:

1. Derive the ridge normal equation and show that

$$(X^T X + \lambda I) \hat{\theta}_{\text{ridge}} = X^T y.$$

2. Explain why practitioners often choose not to penalise the intercept term.
3. Predict how increasing $\lambda$ should affect coefficient size, bias, and variance.
4. Use the degree-15 model and compare several values of $\lambda$.


### Your derivation

Write your answer here. Try to connect regularisation to the bias-variance tradeoff from the lecture.


In [ ]:
def fit_ridge(X, y, lam):
    """Fit ridge regression without penalising the intercept term."""
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    # TODO: build an identity matrix with a zero in the top-left entry
    # TODO: solve the ridge linear system
    # Hint: the penalty should apply to the non-intercept coefficients only.
    raise NotImplementedError("Replace this with your ridge solver.")


In [ ]:
degree = 15
X_train = polynomial_design_matrix(x_train, degree)
X_valid = polynomial_design_matrix(x_valid, degree)
X_test = polynomial_design_matrix(x_test, degree)
grid = np.linspace(-1.0, 1.0, 400)
X_grid = polynomial_design_matrix(grid, degree)

lambdas = [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
ridge_results = []

for lam in lambdas:
    # TODO: fit ridge regression for this lambda
    # TODO: compute train and validation predictions
    # TODO: store lambda, theta, train_mse, valid_mse, and coefficient_norm
    pass

assert ridge_results, "Fill in the lambda sweep above before continuing."

best_entry = min(ridge_results, key=lambda item: item['valid_mse'])
best_theta = best_entry['theta']
test_mse = mse(y_test, X_test @ best_theta)

plt.figure(figsize=(8, 5))
plt.scatter(x_train, y_train, color="tab:blue", label="train")
plt.scatter(x_valid, y_valid, color="tab:orange", label="validation")
plt.scatter(x_test, y_test, color="tab:green", label="test")
plt.plot(grid, np.sin(np.pi * grid) + 0.3 * grid ** 2, color="black", linewidth=2, label="true signal")
plt.plot(grid, X_grid @ best_theta, color="tab:red", linewidth=2, label="best ridge model")
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Best ridge model for degree {degree}")
plt.legend()
plt.show()

for entry in ridge_results:
    print(
        f"lambda = {entry['lambda']:>7} | train MSE = {entry['train_mse']:.4f} | validation MSE = {entry['valid_mse']:.4f} | ||w|| = {entry['coefficient_norm']:.4f}"
    )

print()
print(f"Selected lambda: {best_entry['lambda']}")
print(f"Validation MSE of selected model: {best_entry['valid_mse']:.4f}")
print(f"Test MSE of selected model: {test_mse:.4f}")


## Optional extension

- Compare the best ridge model with the best unregularised polynomial model on the test set.
- Repeat the experiment with more training points. Does the degree-15 model still overfit as badly?
- Replace the closed-form fit with `np.linalg.lstsq` and compare the numerical behaviour.
- Standardise the polynomial features before fitting ridge regression. Does the preferred value of $\lambda$ change?


## Take-away questions

1. Why does a Gaussian noise model naturally lead to squared-error loss?
2. Why can a model with extremely low training error still generalise poorly?
3. How does ridge regularisation change the effective complexity of a linear model?
4. In your own words, how are linear regression, overfitting, and regularisation connected?


## Exercise 5: Fisher Information Across Regression Models

Fisher information measures how sharply peaked the likelihood is around the parameter value supported by the data. In this context, large curvature means the data constrain that direction of $\theta$ strongly, while very small eigenvalues indicate weakly constrained directions and potentially unstable estimates.

For a Gaussian linear model with known noise variance, you should find that

$$I(\theta) = \frac{1}{\sigma^2} X^T X.$$

This applies both to straight-line regression and to polynomial regression, because both models are linear in the parameter vector $\theta$.

In this exercise, compare:

- the degree-1, degree-3, degree-9, and degree-15 polynomial models on the curved dataset
- the degree-15 model with and without ridge regularisation

Tasks:

1. Starting from the Gaussian log-likelihood for the linear regression MLE, derive the Fisher information matrix for $\theta$ and show that it reduces to the expression above.
2. Compute the Fisher information matrix for each unregularised model on the training split, using the known synthetic noise level $\sigma = 0.18$.
3. For each matrix, inspect its trace, eigenvalues, and condition number.
4. For ridge regression, compare the ordinary Fisher information with the regularised precision matrix

$$I_{\mathrm{eff}}(\theta) = I(\theta) + \frac{\lambda}{\sigma^2} \Lambda,$$

where $\Lambda$ is diagonal, with a zero for the intercept term and ones for the remaining coefficients.

5. Explain why a high-degree model can have a larger trace but still be badly conditioned.
6. Explain why ridge improves numerical stability even though it does not add new information from the data likelihood.


### Your derivation

Write your answer here. Start from the Gaussian log-likelihood for the linear regression MLE, then connect the resulting Hessian to the interpretation of Fisher information as local likelihood curvature. A good answer should also distinguish clearly between Fisher information from the likelihood and the extra curvature introduced by the ridge penalty.


In [ ]:
sigma2_curve = 0.18 ** 2


def fisher_information_gaussian(X, sigma2):
    """Return the Fisher information for y ~ N(X theta, sigma2 I)."""
    X = np.asarray(X)

    # TODO: return (X.T @ X) / sigma2
    raise NotImplementedError("Implement the Fisher information matrix.")


def ridge_precision_gaussian(X, sigma2, lam, penalise_intercept=False):
    """Return Fisher information plus the ridge precision contribution."""
    X = np.asarray(X)

    # TODO: build a diagonal penalty matrix
    # TODO: if penalise_intercept is False, leave the intercept unpenalised
    # TODO: return fisher_information_gaussian(X, sigma2) + (lam / sigma2) * penalty
    raise NotImplementedError("Implement the regularised precision matrix.")


def information_summary(matrix, tol=1e-12):
    """Summarise the scale and conditioning of an information matrix."""
    eigvals = np.linalg.eigvalsh(matrix)

    # TODO: compute trace, min eigenvalue, max eigenvalue, rank, and condition number
    raise NotImplementedError("Return a summary dictionary for the matrix.")


In [ ]:
degrees = [1, 3, 9, 15]
info_results = []

for degree in degrees:
    # TODO: build the training design matrix for this degree
    # TODO: compute the Fisher information matrix and its summary
    # TODO: append degree, fisher matrix, eigenvalues, and summary statistics
    pass

assert info_results, "Fill in the loop above before continuing."

ridge_lambdas = [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
ridge_info_results = []
X_degree15 = polynomial_design_matrix(x_train, 15)
plain_info_degree15 = fisher_information_gaussian(X_degree15, sigma2_curve)

for lam in ridge_lambdas:
    # TODO: compute the regularised precision for this lambda
    # TODO: append lambda, eigenvalues, and summary statistics
    pass

assert ridge_info_results, "Fill in the ridge loop above before continuing."
info_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for entry in info_results:
    axes[0].plot(
        np.arange(1, len(entry['eigvals']) + 1),
        entry['eigvals'],
        marker='o',
        label=f"degree {entry['degree']}",
    )

axes[0].set_yscale('log')
axes[0].set_xlabel('eigenvalue index')
axes[0].set_ylabel('eigenvalue')
axes[0].set_title('Unregularised Fisher information spectra')
axes[0].legend()

for entry in ridge_info_results:
    axes[1].plot(
        np.arange(1, len(entry['eigvals']) + 1),
        entry['eigvals'],
        marker='o',
        label=f"lambda={entry['lambda']}",
    )

axes[1].set_yscale('log')
axes[1].set_xlabel('eigenvalue index')
axes[1].set_ylabel('eigenvalue')
axes[1].set_title('Degree-15 regularised precision spectra')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Unregularised models")
for entry in info_results:
    print(
        f"degree = {entry['degree']:>2} | trace = {entry['trace']:.3f} | "
        f"min eig = {entry['min_eig']:.6e} | max eig = {entry['max_eig']:.3f} | "
        f"rank = {entry['rank']:>2} | cond = {entry['condition_number']:.3e}"
    )

print()
print("Degree-15 ridge comparison")
print("Note: ridge does not change the likelihood Fisher information; it changes the effective precision.")
for entry in ridge_info_results:
    print(
        f"lambda = {entry['lambda']:>6} | trace = {entry['trace']:.3f} | "
        f"min eig = {entry['min_eig']:.6e} | max eig = {entry['max_eig']:.3f} | "
        f"rank = {entry['rank']:>2} | cond = {entry['condition_number']:.3e}"
    )


### Reflection questions

- Which model has the largest trace? Which has the best conditioning?
- Why does the degree-15 model have more total curvature than the degree-1 model, yet still behave less stably?
- What happens to the smallest eigenvalues when you add ridge regularisation?
- Why is it more accurate to call the ridge quantity an effective precision matrix rather than a Fisher information matrix from the likelihood alone?
